In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/industrial-machine-condition-classification-mtech/sample_submission.csv
/kaggle/input/competitions/industrial-machine-condition-classification-mtech/student_dataset.csv
/kaggle/input/competitions/industrial-machine-condition-classification-mtech/test.csv


In [2]:

import os
import warnings
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings("ignore")

SEED = 42

# --------------------------------------------------
# 1. FIND THE COMPETITION FILES
# --------------------------------------------------
# Kaggle mounts competition files under /kaggle/input/
INPUT_DIR = "/kaggle/input/competitions/industrial-machine-condition-classification-mtech"

print("Available Kaggle input folders:")
print(os.listdir(INPUT_DIR))

def find_csv(filename_candidates):
    for root, dirs, files in os.walk(INPUT_DIR):
        for filename in filename_candidates:
            if filename in files:
                return os.path.join(root, filename)
    raise FileNotFoundError(
        f"Could not find any of these files: {filename_candidates}. "
        "Check the competition files under /kaggle/input/."
    )

TRAIN_PATH = find_csv(["student_dataset.csv", "student.csv"])
TEST_PATH = find_csv(["test.csv"])

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("\nTraining data shape:", train.shape)
print("Test data shape:", test.shape)
print("\nTraining columns:", train.columns.tolist())

# --------------------------------------------------
# 2. SEPARATE FEATURES AND TARGET
# --------------------------------------------------
TARGET = "condition"
ID = "unit_id"

NUM = [
    "ambient_temp", "process_temp", "coolant_flow",
    "shaft_speed", "drive_torque", "power_load",
    "component_wear", "thermal_stress",
    "mechanical_load", "vibration_index"
]

CAT = ["unit_type"]

required_train = NUM + CAT + [TARGET, ID]
required_test = NUM + CAT + [ID]

missing_train = set(required_train) - set(train.columns)
missing_test = set(required_test) - set(test.columns)

if missing_train:
    raise ValueError(f"Missing training columns: {missing_train}")
if missing_test:
    raise ValueError(f"Missing test columns: {missing_test}")

X = train.drop(columns=[ID, TARGET])
y = train[TARGET]
X_test = test.drop(columns=[ID])

print("\nClass distribution:")
print(y.value_counts().sort_index())

# --------------------------------------------------
# 3. FEATURE ENGINEERING
# --------------------------------------------------
def add_features(df):
    df = df.copy()

    df["temp_diff"] = (
        df["process_temp"] - df["ambient_temp"]
    )

    df["pw"] = df["drive_torque"] * df["shaft_speed"]

    df["wear_torque"] = (
        df["component_wear"] * df["drive_torque"]
    )

    df["cool_ratio"] = (
        df["temp_diff"] / (df["coolant_flow"] + 1e-6)
    )

    return df

NUM_FE = NUM + [
    "temp_diff", "pw", "wear_torque", "cool_ratio"
]

# --------------------------------------------------
# 4. PREPROCESSING PIPELINE
# --------------------------------------------------
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, NUM_FE),
    ("cat", categorical_pipeline, CAT)
])

model_pipeline = Pipeline([
    ("features", FunctionTransformer(add_features)),
    ("preprocessor", preprocessor),
    ("clf", DecisionTreeClassifier(random_state=SEED))
])

# --------------------------------------------------
# 5. STRATIFIED CROSS-VALIDATION
# --------------------------------------------------
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

# --------------------------------------------------
# 6. HYPERPARAMETER TUNING
# --------------------------------------------------
param_grid = {
    "clf__max_depth": [3, 5, 6, 8, 10, 12, None],
    "clf__min_samples_leaf": [1, 2, 5, 10, 20],
    "clf__min_samples_split": [2, 5, 10, 20],
    "clf__criterion": ["gini", "entropy", "log_loss"],
    "clf__class_weight": [None, "balanced"]
}

grid = GridSearchCV(
    estimator=model_pipeline,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True
)

print("\nTraining and tuning Decision Tree...")
grid.fit(X, y)

best_model = grid.best_estimator_

print("\nBest cross-validation accuracy:",
      round(grid.best_score_, 5))
print("Best parameters:")
print(grid.best_params_)

# --------------------------------------------------
# 7. CROSS-VALIDATED CLASSIFICATION REPORT
# --------------------------------------------------
print("\nGenerating cross-validated predictions...")

cv_predictions = cross_val_predict(
    best_model, X, y, cv=cv, n_jobs=-1
)

print("\nCross-validated accuracy:",
      round(accuracy_score(y, cv_predictions), 5))

print("\nClassification report:")
print(classification_report(
    y,
    cv_predictions,
    labels=[0, 1, 2],
    target_names=[
        "Normal",
        "Thermal Failure",
        "Mechanical Failure"
    ],
    zero_division=0
))

# --------------------------------------------------
# 8. PREDICT THE HIDDEN TEST SET
# --------------------------------------------------
test_predictions = best_model.predict(X_test)

# --------------------------------------------------
# 9. CREATE KAGGLE SUBMISSION
# --------------------------------------------------
submission = pd.DataFrame({
    "unit_id": test[ID],
    "condition": test_predictions.astype(int)
})

# Verify the submission format
assert list(submission.columns) == ["unit_id", "condition"]
assert len(submission) == len(test)
assert submission["unit_id"].equals(test[ID])
assert submission["condition"].isin([0, 1, 2]).all()

OUTPUT_PATH = "/kaggle/working/submission.csv"
submission.to_csv(OUTPUT_PATH, index=False)

print("\nSubmission file created successfully!")
print("Location:", OUTPUT_PATH)
print("Submission shape:", submission.shape)
print("\nFirst 10 predictions:")
print(submission.head(10))

print("\nPredicted class distribution:")
print(submission["condition"].value_counts().sort_index())

print("\nSubmission preview:")
print(submission.to_csv(index=False)[:500])


Available Kaggle input folders:
['sample_submission.csv', 'student_dataset.csv', 'test.csv']

Training data shape: (16000, 13)
Test data shape: (4000, 12)

Training columns: ['unit_id', 'unit_type', 'ambient_temp', 'process_temp', 'coolant_flow', 'shaft_speed', 'drive_torque', 'power_load', 'component_wear', 'thermal_stress', 'mechanical_load', 'vibration_index', 'condition']

Class distribution:
condition
0    8000
1    4000
2    4000
Name: count, dtype: int64

Training and tuning Decision Tree...
Fitting 5 folds for each of 840 candidates, totalling 4200 fits

Best cross-validation accuracy: 0.85288
Best parameters:
{'clf__class_weight': None, 'clf__criterion': 'gini', 'clf__max_depth': 10, 'clf__min_samples_leaf': 10, 'clf__min_samples_split': 2}

Generating cross-validated predictions...

Cross-validated accuracy: 0.85288

Classification report:
                    precision    recall  f1-score   support

            Normal       0.86      0.94      0.90      8000
   Thermal Failur